[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/C.%20Transportation%2C%20Routing%20%26%20Freight%20Management%20%28The%20Physical%20Internet%29/079.%20The%20Freight%20Carrier%20Selection%20%26%20Bidding%20Problem/P79-Tier-7_executed.ipynb)

---


# 79. The Freight Carrier Selection and Bidding Problem
## Tier 7: Human–AI Partnership (The Centaur Model)

### Purpose and scope
This notebook illustrates how a human logistics expert and an AI analyst can combine evaluations when selecting among freight carriers. Each carrier receives a human score, an AI score, a trust score, a risk assessment, and a final partnership score. The carrier with the highest final score is selected; its bid price is then used to calculate the cost of the example shipment.

The model is a decision-support demonstration, not a trained predictive system or a procurement recommendation. Historical performance records, human risk assessments, and trust-building cycles are simulated. The carrier bids and shipment demands are illustrative inputs. The model does not enforce capacity, lane eligibility, service-level, or budget constraints, and it does not optimize assignments across lanes. Its final-score ranking and its shipment-cost calculation are therefore separate steps, not a cost-minimizing optimization model.

### Modeling assumptions
- Human judgment contributes relationship knowledge, strategic assessment, and an assessment of risk exposure.
- The AI score summarizes synthetic historical performance, recent trends, and a simple predictive component. Its confidence is based on data completeness and a fixed illustrative model-accuracy value.
- Scores and confidence values are heuristic quantities on a roughly zero-to-one scale; they are not calibrated probabilities.
- Human and AI risk assessments are expressed as risk exposure: larger values indicate greater risk. Human stability assessments are converted to exposure by subtracting their average from one.
- The partnership combines the two scores using trust-dependent base weights, adjusted for the two confidence values and normalized to sum to one. Risk then reduces the combined score.
- The trust-building and financial analyses are illustrative simulations based on explicit assumptions, not empirical evidence of real-world partnership benefits.

### Method at a glance
For carrier \(j\), let \(H_j\) be the human score, \(A_j\) the AI score, \(T_j\) the trust score, and \(R_j\) the combined risk exposure. First choose base weights \(b_{H,j}\) and \(b_{A,j}\) according to whether trust exceeds the trust threshold. Adjust those weights using human and AI confidence \(c_{H,j}\) and \(c_{A,j}\):

$$
\widetilde{w}_{H,j}=b_{H,j}c_{H,j},\qquad
\widetilde{w}_{A,j}=b_{A,j}c_{A,j},
$$

$$
w_{H,j}=\frac{\widetilde{w}_{H,j}}{\widetilde{w}_{H,j}+\widetilde{w}_{A,j}},\qquad
w_{A,j}=\frac{\widetilde{w}_{A,j}}{\widetilde{w}_{H,j}+\widetilde{w}_{A,j}}.
$$

The combined score, combined risk, risk adjustment, and final score are

$$
C_j=w_{H,j}H_j+w_{A,j}A_j,\qquad
R_j=w_{H,j}R_{H,j}+w_{A,j}R_{A,j},
$$

$$
Q_j=1-0.1R_j,\qquad
F_j=C_jQ_j.
$$

Here \(R_{H,j}\) and \(R_{A,j}\) are the human and AI risk exposures. The weights sum to one, so \(C_j\) and \(R_j\) are weighted averages on their respective scales. The factor \(Q_j\) applies a modest risk penalty. The code implements these steps in `CentaurPartnership.evaluate_carrier_partnership`; the selected carrier is the one with the largest `final_score`. Ties are possible, and Python's `max` returns the first maximum encountered in the carrier ordering.

### What to look for
- The separate human and AI explanations, and how their scores and confidences affect the partnership result.
- The trust threshold's role in choosing the base weights; confidence adjustment then changes the final normalized weights.
- The distinction between a high score and a high confidence: confidence affects the weighting and is reported separately.
- The limitations of comparing illustrative benchmarks or simulated ROI calculations as if they were measured outcomes.

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import random
import seaborn as sns

# A fixed seed makes the synthetic data and simulated cycles reproducible.
np.random.seed(42)
random.seed(42)

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

## 1. Representing the human and AI evaluators

The human expert and AI analyst are represented as separate components because they use different evidence and produce different explanations. The expert uses relationship knowledge and simulated strategic and risk assessments. The analyst uses a synthetic 24-month performance history for each carrier, calculates recent-versus-overall trends, and combines performance, trends, and a simple prediction.

A score is a modeled preference measure, not a guaranteed outcome. In particular, the AI's data and predictions here are generated for teaching purposes; they do not come from operational carrier records. The bid data are passed through the evaluation interface for a consistent decision workflow, but the scoring rules in this demonstration do not use bid price or capacity.

In [ ]:
@dataclass
class HumanExpert:
    """A logistics expert with relationship knowledge and domain experience."""

    id: int
    name: str
    experience_years: int
    expertise_areas: List[str]
    relationship_knowledge: Dict[str, float]
    strategic_insights: Dict[str, str]
    risk_tolerance: float
    confidence_level: float

    def evaluate_carrier_human(
        self, carrier_name: str, carrier_data: Dict
    ) -> Dict:
        """Return a heuristic human evaluation and its explanation."""
        relationship_score = self.relationship_knowledge.get(carrier_name, 0.5)

        strategic_factors = {
            "market_position": 0.20,
            "innovation_capability": 0.15,
            "flexibility": 0.15,
            "long_term_partnership": 0.20,
            "reliability_under_pressure": 0.15,
            "cost_transparency": 0.15,
        }

        # Simulated factor assessments represent uncertain expert judgment.
        strategic_score = 0.0
        for weight in strategic_factors.values():
            base_score = np.random.normal(0.7, 0.1)
            experience_boost = min(0.1, self.experience_years / 100)
            factor_score = np.clip(base_score + experience_boost, 0.3, 1.0)
            strategic_score += factor_score * weight

        stability_assessments = {
            "financial_stability": np.random.normal(0.8, 0.1),
            "operational_consistency": np.random.normal(0.75, 0.08),
            "market_volatility_exposure": np.random.normal(0.7, 0.12),
        }
        mean_stability = float(np.clip(np.mean(list(stability_assessments.values())), 0, 1))
        risk_assessment = 1.0 - mean_stability

        if self.risk_tolerance < 0.3:
            risk_penalty_factor = 1.1
        elif self.risk_tolerance > 0.7:
            risk_penalty_factor = 0.9
        else:
            risk_penalty_factor = 1.0

        # Lower exposure raises the risk contribution; risk tolerance scales its penalty.
        human_score = (
            relationship_score * 0.4
            + strategic_score * 0.4
            + (1.0 - risk_assessment * risk_penalty_factor) * 0.2
        )

        return {
            "human_score": float(human_score),
            "relationship_score": float(relationship_score),
            "strategic_score": float(strategic_score),
            "risk_assessment": float(risk_assessment),
            "confidence": float(self.confidence_level),
            "reasoning": self._generate_reasoning(carrier_name),
        }

    def _generate_reasoning(self, carrier_name: str) -> str:
        """Summarize relationship and experience considerations."""
        reasoning_points = []
        relationship_score = self.relationship_knowledge.get(carrier_name, 0.5)

        if relationship_score > 0.8:
            reasoning_points.append(f"Excellent long-term relationship with {carrier_name}")
        elif relationship_score > 0.6:
            reasoning_points.append(f"Good working relationship with {carrier_name}")
        elif relationship_score < 0.4:
            reasoning_points.append(f"Limited experience with {carrier_name}")

        if self.experience_years > 15:
            reasoning_points.append(
                f"Based on {self.experience_years} years of industry experience"
            )

        if self.risk_tolerance < 0.3:
            reasoning_points.append("Prefers conservative, reliable partners")
        elif self.risk_tolerance > 0.7:
            reasoning_points.append("Is open to innovative, higher-risk options")

        return "; ".join(reasoning_points)


@dataclass
class AIAnalyst:
    """An illustrative analyst using synthetic history and simple trend rules."""

    id: int
    name: str
    model_version: str
    data_sources: List[str]
    analytical_capabilities: List[str]
    confidence_threshold: float

    def __post_init__(self):
        self.historical_data = self._generate_historical_data()
        self.model_accuracy = 0.87

    def _generate_historical_data(self) -> pd.DataFrame:
        """Create bounded synthetic monthly performance data for three carriers."""
        carriers = ["Carrier 1", "Carrier 2", "Carrier 3"]
        rows = []
        metric_specs = {
            "on_time_performance": (0.85, 0.08),
            "cost_efficiency": (0.78, 0.06),
            "service_quality": (0.82, 0.07),
            "compliance_rate": (0.94, 0.04),
            "flexibility_score": (0.76, 0.09),
        }

        for carrier in carriers:
            for month in range(24):
                row = {"carrier": carrier, "month": month}
                for metric, (mean, standard_deviation) in metric_specs.items():
                    row[metric] = float(
                        np.clip(np.random.normal(mean, standard_deviation), 0, 1)
                    )
                rows.append(row)

        return pd.DataFrame(rows)

    def analyze_carrier_ai(self, carrier_name: str, carrier_data: Dict) -> Dict:
        """Calculate a heuristic score, confidence, and explainable evidence."""
        carrier_hist = self.historical_data[
            self.historical_data["carrier"] == carrier_name
        ]
        if carrier_hist.empty:
            raise ValueError(f"No historical data available for {carrier_name}.")

        metric_names = [
            "on_time_performance",
            "cost_efficiency",
            "service_quality",
            "compliance_rate",
            "flexibility_score",
        ]
        performance_metrics = {
            metric: float(carrier_hist[metric].mean()) for metric in metric_names
        }

        recent_performance = carrier_hist.tail(6)
        trend_metrics = {}
        for metric, overall_average in performance_metrics.items():
            recent_average = float(recent_performance[metric].mean())
            trend = (recent_average - overall_average) / overall_average
            trend_metrics[f"{metric}_trend"] = float(trend)

        predictive_scores = self._predictive_analysis(
            carrier_name, performance_metrics, trend_metrics
        )
        data_quality = len(carrier_hist) / 24
        model_confidence = self.model_accuracy * data_quality

        ai_score = (
            np.mean(list(performance_metrics.values())) * 0.6
            + np.mean([max(0, value) for value in trend_metrics.values()]) * 0.2
            + predictive_scores["overall_prediction"] * 0.2
        )

        return {
            "ai_score": float(ai_score),
            "performance_metrics": performance_metrics,
            "trend_metrics": trend_metrics,
            "predictive_scores": predictive_scores,
            "confidence": float(model_confidence),
            "reasoning": self._generate_ai_reasoning(
                carrier_name, performance_metrics, trend_metrics
            ),
        }

    def _predictive_analysis(
        self, carrier_name: str, performance_metrics: Dict, trend_metrics: Dict
    ) -> Dict:
        """Project a simple overall score from current metrics and trends."""
        base_prediction = np.mean(list(performance_metrics.values()))
        trend_adjustment = np.mean(
            [np.clip(value, -0.2, 0.2) for value in trend_metrics.values()]
        )

        return {
            "overall_prediction": float(
                np.clip(base_prediction + trend_adjustment, 0.3, 1.0)
            ),
            "reliability_prediction": float(
                performance_metrics["on_time_performance"]
                + trend_metrics.get("on_time_performance_trend", 0)
            ),
            "cost_prediction": float(
                performance_metrics["cost_efficiency"]
                + trend_metrics.get("cost_efficiency_trend", 0)
            ),
            "risk_assessment": float(1 - performance_metrics["compliance_rate"]),
        }

    def _generate_ai_reasoning(
        self, carrier_name: str, performance_metrics: Dict, trend_metrics: Dict
    ) -> str:
        """Report the strongest and weakest metrics and notable trends."""
        reasoning_points = []
        best_metric = max(performance_metrics, key=performance_metrics.get)
        worst_metric = min(performance_metrics, key=performance_metrics.get)

        reasoning_points.append(
            f"Strongest in {best_metric.replace('_', ' ')}: "
            f"{performance_metrics[best_metric]:.3f}"
        )
        reasoning_points.append(
            f"Needs improvement in {worst_metric.replace('_', ' ')}: "
            f"{performance_metrics[worst_metric]:.3f}"
        )

        improving_metrics = [
            key.replace("_trend", "")
            for key, value in trend_metrics.items()
            if value > 0.05
        ]
        if improving_metrics:
            reasoning_points.append(
                f"Improving trends in: {', '.join(improving_metrics)}"
            )

        month_count = len(
            self.historical_data[self.historical_data["carrier"] == carrier_name]
        )
        reasoning_points.append(f"Analysis based on {month_count} months of data")
        return "; ".join(reasoning_points)

## 2. Combining evaluations through the partnership

A trust score summarizes agreement between the human and AI evaluations. Agreement is assessed for overall scores, confidence, and risk exposure, with weights of 0.4, 0.3, and 0.3. When trust exceeds the configured threshold, the base human and AI weights are balanced; otherwise, the base weights favor human judgment. The confidence adjustment then reflects the relative confidence of the two evaluators.

The confidence-adjusted weights must be normalized: without normalization, they would not necessarily sum to one, and the weighted score would be unintentionally scaled down. After combining the scores and risk exposures, the model applies the risk multiplier and assigns a recommendation category.

In [ ]:
@dataclass
class CentaurPartnership:
    """Combine human and AI evaluations using trust and confidence."""

    human_expert: HumanExpert
    ai_analyst: AIAnalyst
    collaboration_model: str
    trust_threshold: float

    def evaluate_carrier_partnership(
        self, carrier_name: str, carrier_data: Dict
    ) -> Dict:
        """Return the combined carrier score, risk, weights, and explanations."""
        human_eval = self.human_expert.evaluate_carrier_human(
            carrier_name, carrier_data
        )
        ai_eval = self.ai_analyst.analyze_carrier_ai(carrier_name, carrier_data)
        trust_score = self._calculate_trust_score(human_eval, ai_eval)

        if trust_score > self.trust_threshold:
            base_human_weight = 0.5
            base_ai_weight = 0.5
        else:
            base_human_weight = 0.7
            base_ai_weight = 0.3

        human_confidence = human_eval["confidence"]
        ai_confidence = ai_eval["confidence"]
        weighted_human_confidence = base_human_weight * human_confidence
        weighted_ai_confidence = base_ai_weight * ai_confidence
        total_weight = weighted_human_confidence + weighted_ai_confidence
        if total_weight <= 0:
            raise ValueError("At least one evaluator must have positive confidence.")

        human_weight = weighted_human_confidence / total_weight
        ai_weight = weighted_ai_confidence / total_weight

        combined_score = (
            human_eval["human_score"] * human_weight
            + ai_eval["ai_score"] * ai_weight
        )
        human_risk = human_eval["risk_assessment"]
        ai_risk = ai_eval["predictive_scores"]["risk_assessment"]
        combined_risk = human_risk * human_weight + ai_risk * ai_weight

        risk_adjustment = 1.0 - combined_risk * 0.1
        final_score = combined_score * risk_adjustment
        overall_confidence = (
            human_confidence * human_weight + ai_confidence * ai_weight
        ) * trust_score

        return {
            "final_score": float(final_score),
            "human_score": float(human_eval["human_score"]),
            "ai_score": float(ai_eval["ai_score"]),
            "human_weight": float(human_weight),
            "ai_weight": float(ai_weight),
            "trust_score": float(trust_score),
            "risk_assessment": float(combined_risk),
            "risk_adjustment": float(risk_adjustment),
            "overall_confidence": float(overall_confidence),
            "human_reasoning": human_eval["reasoning"],
            "ai_reasoning": ai_eval["reasoning"],
            "partnership_recommendation": self._generate_partnership_recommendation(
                final_score, overall_confidence
            ),
        }

    def _calculate_trust_score(self, human_eval: Dict, ai_eval: Dict) -> float:
        """Measure agreement in scores, confidence, and risk exposure."""
        score_difference = abs(human_eval["human_score"] - ai_eval["ai_score"])
        score_alignment = 1.0 - min(score_difference, 1.0)

        confidence_difference = abs(
            human_eval["confidence"] - ai_eval["confidence"]
        )
        confidence_alignment = 1.0 - min(confidence_difference, 1.0)

        human_risk = human_eval["risk_assessment"]
        ai_risk = ai_eval["predictive_scores"]["risk_assessment"]
        risk_difference = abs(human_risk - ai_risk)
        risk_alignment = 1.0 - min(risk_difference, 1.0)

        trust_score = (
            score_alignment * 0.4
            + confidence_alignment * 0.3
            + risk_alignment * 0.3
        )
        return float(trust_score)

    def _generate_partnership_recommendation(
        self, final_score: float, confidence: float
    ) -> str:
        """Translate the score and confidence into a decision category."""
        if final_score > 0.8 and confidence > 0.8:
            return "STRONG RECOMMEND - High confidence, excellent fit"
        if final_score > 0.7 and confidence > 0.6:
            return "RECOMMEND - Good fit with reasonable confidence"
        if final_score > 0.6 and confidence > 0.5:
            return "CONSIDER - Acceptable option, monitor closely"
        if final_score > 0.5:
            return "CAUTION - Below threshold, specific circumstances only"
        return "AVOID - High risk, poor fit"


def initialize_centaur_system() -> CentaurPartnership:
    """Create the expert, analyst, and integrated partnership configuration."""
    human_expert = HumanExpert(
        id=1,
        name="Sarah Chen",
        experience_years=18,
        expertise_areas=["Carrier Relations", "Risk Management", "Strategic Sourcing"],
        relationship_knowledge={
            "Carrier 1": 0.85,
            "Carrier 2": 0.65,
            "Carrier 3": 0.45,
        },
        strategic_insights={
            "Carrier 1": "Innovative, flexible, strong growth potential",
            "Carrier 2": "Reliable, conservative, stable performance",
            "Carrier 3": "Cost-effective, expanding service area",
        },
        risk_tolerance=0.4,
        confidence_level=0.82,
    )

    ai_analyst = AIAnalyst(
        id=1,
        name="LogisticsAI-2.0",
        model_version="v2.3.1",
        data_sources=[
            "Historical Performance",
            "Market Data",
            "Financial Reports",
            "Customer Feedback",
        ],
        analytical_capabilities=[
            "Predictive Analytics",
            "Risk Assessment",
            "Performance Optimization",
            "Trend Analysis",
        ],
        confidence_threshold=0.75,
    )

    partnership = CentaurPartnership(
        human_expert=human_expert,
        ai_analyst=ai_analyst,
        collaboration_model="integrated",
        trust_threshold=0.7,
    )
    return partnership


centaur_system = initialize_centaur_system()
print("Human-AI Partnership System Initialized")
print(
    f"Human Expert: {centaur_system.human_expert.name} "
    f"({centaur_system.human_expert.experience_years} years of experience)"
)
print(
    f"AI Analyst: {centaur_system.ai_analyst.name} "
    f"({centaur_system.ai_analyst.model_version})"
)
print(f"Collaboration Model: {centaur_system.collaboration_model}")
print(f"Trust Threshold: {centaur_system.trust_threshold:.1f}")
print(
    "Human Expertise: "
    + ", ".join(centaur_system.human_expert.expertise_areas)
)
print(
    "AI Capabilities: "
    + ", ".join(centaur_system.ai_analyst.analytical_capabilities)
)

## 3. Evaluate the bidding scenario and select a carrier

Each carrier is evaluated using the same partnership model. The winner is chosen by final score, not by bid price. The cost calculation that follows multiplies the winning carrier's bid price by total demand; it does not alter the score-based selection or verify that the carrier has enough capacity for each lane.

In [ ]:
def centaur_carrier_selection(
    partnership: CentaurPartnership,
) -> Tuple[Dict, Dict]:
    """Evaluate every carrier in the illustrative bidding scenario."""
    carriers_data = {
        "Carrier 1": {
            "bid_price": 500,
            "capacity": 150,
            "reliability": 95,
            "service_quality": 90,
            "coverage": "National",
            "technology_level": "Advanced",
        },
        "Carrier 2": {
            "bid_price": 480,
            "capacity": 120,
            "reliability": 88,
            "service_quality": 92,
            "coverage": "Regional",
            "technology_level": "Moderate",
        },
        "Carrier 3": {
            "bid_price": 520,
            "capacity": 200,
            "reliability": 92,
            "service_quality": 87,
            "coverage": "International",
            "technology_level": "Advanced",
        },
    }

    evaluations = {}
    print("Human-AI Partnership Carrier Evaluation")
    print("=" * 60)

    for carrier_name, carrier_data in carriers_data.items():
        evaluation = partnership.evaluate_carrier_partnership(
            carrier_name, carrier_data
        )
        evaluations[carrier_name] = evaluation

        print(f"\nEvaluating {carrier_name}:")
        print(f"  Final Score: {evaluation['final_score']:.3f}")
        print(
            f"  Human Score: {evaluation['human_score']:.3f} "
            f"(Weight: {evaluation['human_weight']:.2f})"
        )
        print(
            f"  AI Score: {evaluation['ai_score']:.3f} "
            f"(Weight: {evaluation['ai_weight']:.2f})"
        )
        print(f"  Trust Score: {evaluation['trust_score']:.3f}")
        print(f"  Risk Exposure: {evaluation['risk_assessment']:.3f}")
        print(f"  Overall Confidence: {evaluation['overall_confidence']:.3f}")
        print(f"  Recommendation: {evaluation['partnership_recommendation']}")
        print(f"  Human Reasoning: {evaluation['human_reasoning']}")
        print(f"  AI Reasoning: {evaluation['ai_reasoning']}")

    return evaluations, carriers_data


centaur_evaluations, carriers_data = centaur_carrier_selection(centaur_system)
best_carrier = max(
    centaur_evaluations,
    key=lambda name: centaur_evaluations[name]["final_score"],
)
best_evaluation = centaur_evaluations[best_carrier]

demand = [100, 80]
total_cost = carriers_data[best_carrier]["bid_price"] * sum(demand)

print("\n" + "=" * 60)
print("CENTAUR PARTNERSHIP DECISION")
print("=" * 60)
print(f"Selected Carrier: {best_carrier}")
print(f"Final Score: {best_evaluation['final_score']:.3f}")
print(f"Overall Confidence: {best_evaluation['overall_confidence']:.3f}")
print(f"Recommendation: {best_evaluation['partnership_recommendation']}")
print(f"Shipment Cost: ${total_cost:,.2f}")
print(f"Risk Adjustment Factor: {best_evaluation['risk_adjustment']:.3f}")

## 4. Visualize the carrier evaluations

The plots show distinct aspects of the same evaluations: score components, trust and confidence, normalized decision weights, and the relationship between bid price, risk exposure, and final score. Bubble size and color in the final plot both encode final score; the color bar identifies the color scale.

In [ ]:
def visualize_centaur_results(evaluations: Dict, carriers_data: Dict) -> None:
    """Plot partnership scores, weights, trust, confidence, risk, and bids."""
    if not evaluations:
        raise ValueError("At least one carrier evaluation is required.")

    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle(
        "Human-AI Partnership Analysis", fontsize=16, fontweight="bold"
    )

    carrier_names = list(evaluations)
    final_scores = [evaluations[name]["final_score"] for name in carrier_names]
    human_scores = [evaluations[name]["human_score"] for name in carrier_names]
    ai_scores = [evaluations[name]["ai_score"] for name in carrier_names]
    trust_scores = [evaluations[name]["trust_score"] for name in carrier_names]
    confidence_scores = [
        evaluations[name]["overall_confidence"] for name in carrier_names
    ]

    ax1 = axes[0, 0]
    positions = np.arange(len(carrier_names))
    width = 0.25
    ax1.bar(
        positions - width,
        human_scores,
        width,
        label="Human Score",
        color="lightblue",
        alpha=0.8,
    )
    ax1.bar(
        positions,
        ai_scores,
        width,
        label="AI Score",
        color="lightgreen",
        alpha=0.8,
    )
    ax1.bar(
        positions + width,
        final_scores,
        width,
        label="Final Score",
        color="gold",
        alpha=0.8,
    )
    ax1.set_xlabel("Carrier")
    ax1.set_ylabel("Heuristic Score")
    ax1.set_title("Human, AI, and Final Scores")
    ax1.set_xticks(positions)
    ax1.set_xticklabels(carrier_names)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    score_max = max(max(final_scores), max(human_scores), max(ai_scores))
    ax1.set_ylim(0, max(1.0, score_max * 1.1))

    ax2 = axes[0, 1]
    scatter = ax2.scatter(
        trust_scores,
        confidence_scores,
        s=200,
        c=final_scores,
        cmap="viridis",
        alpha=0.7,
        edgecolors="black",
        linewidth=1.5,
    )
    for index, name in enumerate(carrier_names):
        ax2.annotate(
            name,
            (trust_scores[index], confidence_scores[index]),
            xytext=(5, 5),
            textcoords="offset points",
        )
    ax2.set_xlabel("Trust Score")
    ax2.set_ylabel("Overall Confidence")
    ax2.set_title("Trust and Confidence")
    ax2.grid(True, alpha=0.3)
    ax2.set_xlim(0, 1)
    ax2.set_ylim(0, 1)
    colorbar = fig.colorbar(scatter, ax=ax2)
    colorbar.set_label("Final Score")

    ax3 = axes[1, 0]
    human_weights = [evaluations[name]["human_weight"] for name in carrier_names]
    ai_weights = [evaluations[name]["ai_weight"] for name in carrier_names]
    ax3.bar(
        carrier_names,
        human_weights,
        label="Human Weight",
        color="lightblue",
        alpha=0.8,
    )
    ax3.bar(
        carrier_names,
        ai_weights,
        bottom=human_weights,
        label="AI Weight",
        color="lightgreen",
        alpha=0.8,
    )
    ax3.set_ylabel("Normalized Decision Weight")
    ax3.set_title("Human and AI Weights")
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    ax3.set_ylim(0, 1)
    for index, (human_weight, ai_weight) in enumerate(
        zip(human_weights, ai_weights)
    ):
        ax3.text(
            index,
            human_weight / 2,
            f"{human_weight:.1%}",
            ha="center",
            va="center",
        )
        ax3.text(
            index,
            human_weight + ai_weight / 2,
            f"{ai_weight:.1%}",
            ha="center",
            va="center",
        )

    ax4 = axes[1, 1]
    risks = [evaluations[name]["risk_assessment"] for name in carrier_names]
    bids = [carriers_data[name]["bid_price"] for name in carrier_names]
    bubble_sizes = [max(score, 0) * 1000 for score in final_scores]
    scatter = ax4.scatter(
        bids,
        risks,
        s=bubble_sizes,
        c=final_scores,
        cmap="plasma",
        alpha=0.7,
        edgecolors="black",
        linewidth=1.5,
    )
    for index, name in enumerate(carrier_names):
        ax4.annotate(
            name,
            (bids[index], risks[index]),
            xytext=(5, 5),
            textcoords="offset points",
        )
    ax4.set_xlabel("Bid Price ($ per demand unit)")
    ax4.set_ylabel("Risk Exposure")
    ax4.set_title("Bid, Risk, and Final Score")
    ax4.grid(True, alpha=0.3)
    colorbar = fig.colorbar(scatter, ax=ax4)
    colorbar.set_label("Final Score; bubble size also scales with score")

    fig.tight_layout()
    plt.show()


visualize_centaur_results(centaur_evaluations, carriers_data)

## 5. Assessing partnership results and illustrative benchmarks

The first analysis describes the results of the partnership model for this set of carriers. Human-only and AI-only winners are selected using their respective scores, while the partnership winner is selected using the final partnership score. These rankings do not establish that the partnership has better real-world decision quality: the three scores are heuristic measures, and no independent outcome or ground truth is available.

The comparison table and chart use illustrative benchmark values for other decision approaches. These are assumptions for demonstrating a comparison workflow, not measured results from other notebook tiers. Their quality, explainability, adaptability, and oversight values are not empirically validated, and the cost values may not represent equivalent scopes. Interpret the visual comparison as a template, not evidence that one method outperforms another.

In [ ]:
def analyze_partnership_effectiveness(evaluations: Dict) -> None:
    """Summarize partnership metrics and compare score-based winners."""
    if not evaluations:
        raise ValueError("At least one carrier evaluation is required.")

    trust_values = [item["trust_score"] for item in evaluations.values()]
    confidence_values = [
        item["overall_confidence"] for item in evaluations.values()
    ]
    human_weights = [item["human_weight"] for item in evaluations.values()]
    ai_weights = [item["ai_weight"] for item in evaluations.values()]

    print("=" * 70)
    print("HUMAN-AI PARTNERSHIP EFFECTIVENESS SUMMARY")
    print("=" * 70)
    print(f"Average Trust Score: {np.mean(trust_values):.3f}")
    print(f"Average Overall Confidence: {np.mean(confidence_values):.3f}")
    print(f"Average Human Weight: {np.mean(human_weights):.1%}")
    print(f"Average AI Weight: {np.mean(ai_weights):.1%}")

    print("\nScore alignment by carrier:")
    for carrier_name, evaluation in evaluations.items():
        human_alignment = 1 - abs(
            evaluation["final_score"] - evaluation["human_score"]
        )
        ai_alignment = 1 - abs(
            evaluation["final_score"] - evaluation["ai_score"]
        )
        print(f"  {carrier_name}:")
        print(f"    Alignment with human score: {human_alignment:.3f}")
        print(f"    Alignment with AI score: {ai_alignment:.3f}")
        print(f"    Trust score: {evaluation['trust_score']:.3f}")

    human_dominant = []
    ai_dominant = []
    balanced = []
    for carrier_name, evaluation in evaluations.items():
        if evaluation["human_weight"] > 0.6:
            human_dominant.append(carrier_name)
        elif evaluation["ai_weight"] > 0.6:
            ai_dominant.append(carrier_name)
        else:
            balanced.append(carrier_name)

    print("\nWeight categories:")
    print(f"  Human-dominant: {human_dominant}")
    print(f"  AI-dominant: {ai_dominant}")
    print(f"  Balanced: {balanced}")

    human_only_winner = max(
        evaluations, key=lambda name: evaluations[name]["human_score"]
    )
    ai_only_winner = max(
        evaluations, key=lambda name: evaluations[name]["ai_score"]
    )
    partnership_winner = max(
        evaluations, key=lambda name: evaluations[name]["final_score"]
    )
    print("\nScore-based winners:")
    print(
        f"  Human-only: {human_only_winner} "
        f"(score {evaluations[human_only_winner]['human_score']:.3f})"
    )
    print(
        f"  AI-only: {ai_only_winner} "
        f"(score {evaluations[ai_only_winner]['ai_score']:.3f})"
    )
    print(
        f"  Partnership: {partnership_winner} "
        f"(score {evaluations[partnership_winner]['final_score']:.3f})"
    )
    print(
        "These scores are not independently validated outcome measures; "
        "their differences are not evidence of realized improvement."
    )


def compare_with_other_methods(
    evaluations: Dict, total_cost: float
) -> pd.DataFrame:
    """Display an illustrative comparison against assumed benchmark values."""
    partnership_winner = max(
        evaluations, key=lambda name: evaluations[name]["final_score"]
    )
    partnership_score = evaluations[partnership_winner]["final_score"]

    methods_comparison = {
        "Mathematical Optimization": {
            "cost": 76480,
            "decision_quality": 1.0,
            "explainability": 0.3,
            "adaptability": 0.2,
            "human_oversight": 0.1,
        },
        "Greedy Heuristic": {
            "cost": 85200,
            "decision_quality": 0.7,
            "explainability": 0.8,
            "adaptability": 0.6,
            "human_oversight": 0.3,
        },
        "ML-Enhanced": {
            "cost": 78000,
            "decision_quality": 0.85,
            "explainability": 0.4,
            "adaptability": 0.7,
            "human_oversight": 0.4,
        },
        "Human-AI Partnership": {
            "cost": total_cost,
            "decision_quality": partnership_score,
            "explainability": 0.9,
            "adaptability": 0.8,
            "human_oversight": 0.9,
        },
    }
    comparison_df = pd.DataFrame(methods_comparison).T

    print("\nIllustrative benchmark comparison (assumed values):")
    print(comparison_df.round(3))

    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle(
        "Illustrative Human-AI Partnership Comparison",
        fontsize=16,
        fontweight="bold",
    )
    methods = list(comparison_df.index)
    colors = ["lightgreen", "lightblue", "lightcoral", "gold"]
    metrics = [
        ("cost", "Cost ($)", "Illustrative Cost"),
        ("decision_quality", "Heuristic Score", "Decision-Quality Assumption"),
        ("explainability", "Assumed Score", "Explainability Assumption"),
        ("adaptability", "Assumed Score", "Adaptability Assumption"),
        ("human_oversight", "Assumed Score", "Human-Oversight Assumption"),
    ]

    for axis, (metric, ylabel, title) in zip(axes.flat[:5], metrics):
        axis.bar(methods, comparison_df[metric], color=colors, alpha=0.8)
        axis.set_ylabel(ylabel)
        axis.set_title(title)
        axis.tick_params(axis="x", rotation=35)
        axis.grid(True, alpha=0.3)
        if metric != "cost":
            axis.set_ylim(0, 1)

    overall_axis = axes.flat[5]
    weights = {
        "cost": -0.3,
        "decision_quality": 0.3,
        "explainability": 0.2,
        "adaptability": 0.1,
        "human_oversight": 0.1,
    }
    cost_range = comparison_df["cost"].max() - comparison_df["cost"].min()
    if np.isclose(cost_range, 0.0):
        normalized_costs = pd.Series(0.0, index=comparison_df.index)
    else:
        normalized_costs = (
            comparison_df["cost"].max() - comparison_df["cost"]
        ) / cost_range

    overall_scores = []
    for method in methods:
        score = normalized_costs.loc[method] * abs(weights["cost"])
        for metric in [
            "decision_quality",
            "explainability",
            "adaptability",
            "human_oversight",
        ]:
            score += comparison_df.loc[method, metric] * weights[metric]
        overall_scores.append(score)

    overall_axis.bar(methods, overall_scores, color=colors, alpha=0.8)
    overall_axis.set_ylabel("Weighted Illustrative Score")
    overall_axis.set_title("Weighted Benchmark Assumptions")
    overall_axis.tick_params(axis="x", rotation=35)
    overall_axis.grid(True, alpha=0.3)

    fig.tight_layout()
    plt.show()
    return comparison_df


analyze_partnership_effectiveness(centaur_evaluations)
comparison_df = compare_with_other_methods(centaur_evaluations, total_cost)

## 6. Simulated trust development and illustrative return analysis

The trust simulation generates ten cycles of confidence, trust, and decision weights. It is a prescribed synthetic trajectory with random variation, not a learning algorithm: the evaluation model does not update its parameters from observed decisions. The plotted weights in this simulation follow its own simple cycle rules and are not recalculated by `CentaurPartnership`.

The ROI calculation uses assumed implementation costs, annual spend, efficiency gains, and risk-reduction benefits. Its shipment-cost savings term scales the difference between two example shipment costs to annual spend; this assumes those costs are comparable and representative. The resulting figures are scenario arithmetic, not a forecast or measured ROI. A non-positive net annual benefit does not produce a meaningful positive payback period, so the code reports payback as undefined in that case.

In [ ]:
def simulate_trust_building(
    trust_threshold: float, num_cycles: int = 10
) -> Dict[str, List[float]]:
    """Simulate illustrative trust, confidence, and weight values over cycles."""
    trust_history = []
    confidence_history = []
    human_weight_history = []

    print("=" * 70)
    print("ILLUSTRATIVE TRUST-BUILDING SIMULATION")
    print("=" * 70)

    for cycle in range(num_cycles):
        ai_confidence = min(0.95, 0.75 + cycle * 0.02)
        human_confidence = float(
            np.clip(0.82 + np.random.normal(0, 0.02), 0.7, 0.9)
        )
        base_trust = 0.6 + cycle * 0.03
        trust_score = float(
            np.clip(base_trust + np.random.normal(0, 0.05), 0.3, 1.0)
        )

        if trust_score > 0.8:
            human_weight = 0.4 + cycle * 0.01
        else:
            human_weight = 0.7
        average_confidence = (human_confidence + ai_confidence) / 2

        trust_history.append(trust_score)
        confidence_history.append(average_confidence)
        human_weight_history.append(human_weight)
        print(
            f"Cycle {cycle + 1:2d}: Trust={trust_score:.3f}, "
            f"Confidence={average_confidence:.3f}, "
            f"Human Weight={human_weight:.2f}"
        )

    cycles = range(1, num_cycles + 1)
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.suptitle(
        "Simulated Trust and Decision Weights", fontsize=16, fontweight="bold"
    )

    axes[0].plot(cycles, trust_history, "o-", linewidth=2, color="blue")
    axes[0].axhline(
        y=trust_threshold,
        color="red",
        linestyle="--",
        label=f"Trust Threshold ({trust_threshold:.1f})",
    )
    axes[0].set_xlabel("Decision Cycle")
    axes[0].set_ylabel("Trust Score")
    axes[0].set_title("Trust by Cycle")
    axes[0].grid(True, alpha=0.3)
    axes[0].set_ylim(0, 1)
    axes[0].legend()

    axes[1].plot(cycles, confidence_history, "s-", linewidth=2, color="green")
    axes[1].set_xlabel("Decision Cycle")
    axes[1].set_ylabel("Average Confidence")
    axes[1].set_title("Confidence by Cycle")
    axes[1].grid(True, alpha=0.3)
    axes[1].set_ylim(0, 1)

    axes[2].plot(
        cycles,
        human_weight_history,
        "^-",
        linewidth=2,
        color="orange",
        label="Human Weight",
    )
    axes[2].plot(
        cycles,
        [1 - weight for weight in human_weight_history],
        "--",
        linewidth=2,
        color="purple",
        label="AI Weight",
    )
    axes[2].set_xlabel("Decision Cycle")
    axes[2].set_ylabel("Decision Weight")
    axes[2].set_title("Simulated Weight by Cycle")
    axes[2].grid(True, alpha=0.3)
    axes[2].set_ylim(0, 1)
    axes[2].legend()

    fig.tight_layout()
    plt.show()

    print("\nSimulation summary:")
    print(f"Initial trust: {trust_history[0]:.3f}")
    print(f"Final trust: {trust_history[-1]:.3f}")
    print(f"Change in trust: {trust_history[-1] - trust_history[0]:.3f}")
    print(
        "Weight convergence: "
        + (
            "Near balance"
            if abs(human_weight_history[-1] - 0.5) < 0.1
            else "Not near balance"
        )
    )
    return {
        "trust": trust_history,
        "confidence": confidence_history,
        "human_weight": human_weight_history,
    }


def calculate_partnership_roi(
    best_evaluation: Dict,
    total_cost: float,
    traditional_cost: float = 76480,
) -> Dict[str, Optional[float]]:
    """Calculate scenario-based ROI from explicit illustrative assumptions."""
    ai_system_cost = 100000
    training_cost = 25000
    integration_cost = 35000
    annual_maintenance_cost = 15000
    implementation_cost = ai_system_cost + training_cost + integration_cost

    annual_spend = 2000000
    cost_savings = annual_spend * (
        (traditional_cost - total_cost) / traditional_cost
    )
    efficiency_gains = annual_spend * 0.05
    risk_reduction = annual_spend * 0.02
    annual_benefits = cost_savings + efficiency_gains + risk_reduction
    net_annual_benefit = annual_benefits - annual_maintenance_cost

    first_year_roi = (
        net_annual_benefit / implementation_cost * 100
        if implementation_cost > 0
        else None
    )
    subsequent_year_roi = (
        net_annual_benefit / annual_maintenance_cost * 100
        if annual_maintenance_cost > 0
        else None
    )
    payback_period = (
        implementation_cost / net_annual_benefit
        if net_annual_benefit > 0
        else None
    )

    print("\n" + "=" * 70)
    print("ILLUSTRATIVE HUMAN-AI PARTNERSHIP ROI")
    print("=" * 70)
    print(f"Traditional shipment-cost assumption: ${traditional_cost:,.2f}")
    print(f"Partnership shipment cost: ${total_cost:,.2f}")
    print(f"Cost difference: ${total_cost - traditional_cost:+,.2f}")
    print("\nImplementation-cost assumptions:")
    print(f"  AI system development: ${ai_system_cost:,.2f}")
    print(f"  Training: ${training_cost:,.2f}")
    print(f"  Integration: ${integration_cost:,.2f}")
    print(f"  Total implementation: ${implementation_cost:,.2f}")
    print(f"  Annual maintenance: ${annual_maintenance_cost:,.2f}")
    print("\nAnnual-benefit assumptions:")
    print(f"  Annual transportation spend: ${annual_spend:,.2f}")
    print(f"  Scaled shipment-cost difference: ${cost_savings:,.2f}")
    print(f"  Assumed efficiency gains: ${efficiency_gains:,.2f}")
    print(f"  Assumed risk reduction: ${risk_reduction:,.2f}")
    print(f"  Total assumed annual benefits: ${annual_benefits:,.2f}")
    print(f"  Net benefit after maintenance: ${net_annual_benefit:,.2f}")
    print("\nScenario metrics:")
    print(
        "  First-year ROI: "
        + (f"{first_year_roi:.1f}%" if first_year_roi is not None else "Undefined")
    )
    print(
        "  Subsequent-year ROI: "
        + (
            f"{subsequent_year_roi:.1f}%"
            if subsequent_year_roi is not None
            else "Undefined"
        )
    )
    print(
        "  Payback period: "
        + (
            f"{payback_period:.1f} years"
            if payback_period is not None
            else "Undefined because net annual benefit is not positive"
        )
    )

    if first_year_roi is not None and payback_period is not None:
        if first_year_roi > 20 and payback_period < 3:
            recommendation = "Strong scenario result under the stated assumptions"
        elif first_year_roi > 10 and payback_period < 5:
            recommendation = "Positive scenario result under the stated assumptions"
        elif first_year_roi > 0:
            recommendation = "Positive first-year scenario ROI; review assumptions"
        else:
            recommendation = "Negative first-year scenario ROI"
    else:
        recommendation = "No positive payback conclusion from these assumptions"
    print(f"\nScenario interpretation: {recommendation}")
    print(
        "These calculations are assumption-driven and are not a forecast "
        "or measured return."
    )

    return {
        "first_year_roi_percent": first_year_roi,
        "subsequent_year_roi_percent": subsequent_year_roi,
        "payback_period_years": payback_period,
        "annual_benefits": annual_benefits,
        "net_annual_benefit": net_annual_benefit,
        "decision_score": best_evaluation["final_score"],
    }


trust_simulation = simulate_trust_building(
    centaur_system.trust_threshold, num_cycles=10
)
roi_results = calculate_partnership_roi(best_evaluation, total_cost)

## 7. Positioning, strengths, and limitations

### Why this tier is useful
A human–AI partnership makes room for both quantitative performance analysis and domain knowledge that may not be present in structured data. Explanations, confidence values, and risk assessments make the recommendation easier to inspect. Trust-dependent weighting provides a transparent rule for changing the balance between the two evaluators.

### Strengths
- Makes human judgment and AI analysis separately visible before combining them.
- Provides component scores, confidence, trust, risk exposure, and textual reasoning.
- Uses explicit rules for score combination and recommendation categories.
- Can be adapted to a procurement setting by replacing simulated inputs with validated operational evidence and defining the relevant business constraints.

### Limitations
- Synthetic data and heuristic scores do not demonstrate predictive accuracy or real decision quality.
- The human strategic assessments are simulated and do not use the supplied `strategic_insights` text as a scoring input.
- Bid price, capacity, coverage, and service attributes do not affect the partnership ranking; capacity and lane feasibility are not enforced.
- Trust is a similarity measure, not proof that either evaluator is correct. Confidence is not calibrated against observed forecast accuracy.
- The simulated trust cycles do not update the evaluators or learn from outcomes.
- The benchmark comparison and ROI analysis depend on assumed values and comparability; they should not be treated as empirical performance claims.

### Appropriate uses
This structure can serve as a teaching example or a starting point for a decision-support prototype where human review, explicit reasoning, and risk awareness matter. A real deployment would require validated data, calibrated scores, auditable decision rules, and constraints that match the procurement decision. This notebook intentionally demonstrates the human–AI scoring workflow rather than adding those features.

## 8. Final assessment

The final summary reports the selected carrier and its partnership metrics. Selection is based on the largest heuristic final score. The cost is calculated from the selected carrier's bid and the example demand, so a high-scoring carrier is not necessarily the lowest-cost or feasible carrier. Use the result to inspect how the stated human–AI rules behave on the illustrative inputs, not as evidence of a real procurement outcome.

In [ ]:
print("=" * 70)
print("HUMAN-AI PARTNERSHIP: FINAL SUMMARY")
print("=" * 70)
print(f"Selected Carrier: {best_carrier}")
print(f"Final Partnership Score: {best_evaluation['final_score']:.3f}")
print(f"Overall Confidence: {best_evaluation['overall_confidence']:.3f}")
print(f"Trust Score: {best_evaluation['trust_score']:.3f}")
print(f"Shipment Cost: ${total_cost:,.2f}")
print(f"Recommendation: {best_evaluation['partnership_recommendation']}")

print("\nPartnership components:")
print(f"  Human Weight: {best_evaluation['human_weight']:.1%}")
print(f"  AI Weight: {best_evaluation['ai_weight']:.1%}")
print(f"  Human Score: {best_evaluation['human_score']:.3f}")
print(f"  AI Score: {best_evaluation['ai_score']:.3f}")
print(f"  Combined Risk Exposure: {best_evaluation['risk_assessment']:.3f}")
print(f"  Risk Adjustment Factor: {best_evaluation['risk_adjustment']:.3f}")

assert np.isclose(
    best_evaluation["human_weight"] + best_evaluation["ai_weight"], 1.0
), "Normalized partnership weights must sum to one."
assert best_carrier in carriers_data
assert total_cost == carriers_data[best_carrier]["bid_price"] * sum(demand)
assert best_evaluation["final_score"] == max(
    item["final_score"] for item in centaur_evaluations.values()
)

print("\nInterpretation:")
print(
    "The selected carrier maximizes the partnership score under the "
    "illustrative scoring rules. The score is not an optimized cost, "
    "a calibrated probability, or a guarantee of service performance."
)
print("=" * 70)